In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats


# 0. Configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_Root/
    ├─ Code/
    ├─ Data/
    │  └─ SCB-Mesozoic-Granite.xls
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (
            (path / "Code").exists()
            and (path / "Data").exists()
            and (path / "Result").exists()
        ):
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )


PROJECT_ROOT = find_project_root()

# Output directory from the previous fold-wise preprocessing and
# ratio-feature construction script.
INPUT_ROOT = (
    PROJECT_ROOT
    / "Result"
    / "01_Foldwise_Preprocessing_and_Ratio_Features"
)

# Output directory for this script.
OUT_DIR = (
    PROJECT_ROOT
    / "Result"
    / "02_Foldwise_Normality_Test"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

NON_NUM_COLS = {
    "No.",
    "Sample",
    "Type",
    "Reference"
}

IMPUTATION_METHODS = ["global_mean", "knn"]
N_SPLITS = 5

ALPHA = 0.05
MIN_N = 20


# ============================================================
# 1. Utility functions
# ============================================================

def get_feature_cols(df):
    """
    Get numerical feature columns used for normality testing.

    Non-modeling columns such as sample identifiers and type labels
    are excluded.
    """
    candidate_cols = [
        c for c in df.columns
        if c not in NON_NUM_COLS
    ]

    feature_cols = []

    for c in candidate_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        if pd.api.types.is_numeric_dtype(s):
            feature_cols.append(c)

    return feature_cols


def normality_test_one_feature(x, alpha=0.05, min_n=20):
    """
    Perform normality tests for a single feature.

    Tests used:
    1. D'Agostino K-squared normality test.
    2. Jarque-Bera test.

    Decision rules:
    - If either test has p < alpha, the feature is classified as
      non-normal.
    - If neither test rejects normality, the feature is classified as
      normal-like.
    - If the sample size is too small, the feature is constant, or the
      tests fail, the feature is marked as not tested.
    """
    x = pd.to_numeric(x, errors="coerce")
    x = x.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

    n = len(x)

    mean = float(x.mean()) if n > 0 else np.nan
    std = float(x.std(ddof=1)) if n > 1 else np.nan
    skew = float(stats.skew(x, bias=False)) if n >= 3 else np.nan
    kurt_excess = float(stats.kurtosis(x, fisher=True, bias=False)) if n >= 4 else np.nan

    result = {
        "n_valid": n,
        "mean": mean,
        "std": std,
        "skewness": skew,
        "kurtosis_excess": kurt_excess,
        "K2_stat": np.nan,
        "K2_p": np.nan,
        "JB_stat": np.nan,
        "JB_p": np.nan,
        "p_min": np.nan,
        "alpha": alpha,
        "decision": "Not tested"
    }

    if n < min_n:
        result["decision"] = "Not tested (too few samples)"
        return result

    if pd.isna(std) or std == 0:
        result["decision"] = "Not tested (constant feature)"
        return result

    # D'Agostino K-squared test.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        try:
            k2_stat, k2_p = stats.normaltest(x)
            result["K2_stat"] = float(k2_stat)
            result["K2_p"] = float(k2_p)
        except Exception:
            result["K2_stat"] = np.nan
            result["K2_p"] = np.nan

        # Jarque-Bera test.
        try:
            jb = stats.jarque_bera(x)
            result["JB_stat"] = float(jb.statistic)
            result["JB_p"] = float(jb.pvalue)
        except Exception:
            result["JB_stat"] = np.nan
            result["JB_p"] = np.nan

    p_values = [
        p for p in [result["K2_p"], result["JB_p"]]
        if pd.notna(p)
    ]

    if len(p_values) == 0:
        result["decision"] = "Not tested (test failed)"
        return result

    result["p_min"] = float(np.min(p_values))

    reject_k2 = (
        result["K2_p"] < alpha
        if pd.notna(result["K2_p"])
        else False
    )

    reject_jb = (
        result["JB_p"] < alpha
        if pd.notna(result["JB_p"])
        else False
    )

    if reject_k2 or reject_jb:
        result["decision"] = "Non-normal (reject normality)"
    else:
        result["decision"] = "Normal-like (fail to reject normality)"

    return result


def normality_test_dataframe(df, method, fold, alpha=0.05, min_n=20):
    """
    Perform normality tests for all features in one training-fold dataset.
    """
    feature_cols = get_feature_cols(df)

    rows = []

    for col in feature_cols:
        test_result = normality_test_one_feature(
            df[col],
            alpha=alpha,
            min_n=min_n
        )

        row = {
            "Method": method,
            "Fold": fold,
            "Feature": col
        }

        row.update(test_result)
        rows.append(row)

    res = pd.DataFrame(rows)

    # Sort by decision first and then by p_min in descending order.
    if "p_min" in res.columns:
        res = res.sort_values(
            by=["decision", "p_min", "n_valid"],
            ascending=[True, False, False]
        )

    return res


def summarize_one_fold(res):
    """
    Summarize the normality-test results for one training fold.
    """
    total_features = len(res)

    tested_mask = res["decision"].isin([
        "Non-normal (reject normality)",
        "Normal-like (fail to reject normality)"
    ])

    tested = int(tested_mask.sum())

    normal_like = int(
        (res["decision"] == "Normal-like (fail to reject normality)").sum()
    )

    non_normal = int(
        (res["decision"] == "Non-normal (reject normality)").sum()
    )

    not_tested = total_features - tested

    return {
        "Total_features": total_features,
        "Tested_features": tested,
        "Normal_like_features": normal_like,
        "Non_normal_features": non_normal,
        "Not_tested_features": not_tested,
        "Non_normal_percent_among_tested": (
            non_normal / tested * 100 if tested > 0 else np.nan
        )
    }


def aggregate_across_folds(all_results):
    """
    Aggregate feature-level normality decisions across all folds.
    """
    df_all = pd.concat(all_results, axis=0, ignore_index=True)

    def count_decision(s, decision):
        return int((s == decision).sum())

    agg = (
        df_all
        .groupby(["Method", "Feature"], as_index=False)
        .agg(
            Folds_tested=(
                "decision",
                lambda s: int(s.isin([
                    "Non-normal (reject normality)",
                    "Normal-like (fail to reject normality)"
                ]).sum())
            ),
            Non_normal_folds=(
                "decision",
                lambda s: count_decision(s, "Non-normal (reject normality)")
            ),
            Normal_like_folds=(
                "decision",
                lambda s: count_decision(s, "Normal-like (fail to reject normality)")
            ),
            Not_tested_folds=(
                "decision",
                lambda s: int(s.str.startswith("Not tested").sum())
            ),
            Median_K2_p=("K2_p", "median"),
            Median_JB_p=("JB_p", "median"),
            Median_p_min=("p_min", "median"),
            Median_skewness=("skewness", "median"),
            Median_kurtosis_excess=("kurtosis_excess", "median")
        )
    )

    agg["Non_normal_fold_percent"] = (
        agg["Non_normal_folds"] / agg["Folds_tested"] * 100
    )

    agg["Overall_decision"] = np.where(
        agg["Non_normal_folds"] > 0,
        "Mostly/partly non-normal across folds",
        "Normal-like across tested folds"
    )

    agg = agg.sort_values(
        by=["Method", "Non_normal_fold_percent", "Median_p_min"],
        ascending=[True, False, True]
    )

    return df_all, agg


# ============================================================
# 2. Main function
# ============================================================

def main():
    all_results = []
    fold_summary_records = []

    for method in IMPUTATION_METHODS:
        method_input_dir = INPUT_ROOT / method

        if not method_input_dir.exists():
            raise FileNotFoundError(
                f"Input directory was not found: {method_input_dir}"
            )

        method_out_dir = OUT_DIR / method
        method_out_dir.mkdir(parents=True, exist_ok=True)

        for fold in range(1, N_SPLITS + 1):
            train_path = (
                method_input_dir
                / f"fold_{fold:02d}_train_with_ratios.xlsx"
            )

            if not train_path.exists():
                raise FileNotFoundError(
                    f"Training-fold file was not found: {train_path}"
                )

            df_train = pd.read_excel(train_path)
            df_train.columns = [str(c).strip() for c in df_train.columns]

            res = normality_test_dataframe(
                df_train,
                method=method,
                fold=fold,
                alpha=ALPHA,
                min_n=MIN_N
            )

            out_path = (
                method_out_dir
                / f"fold_{fold:02d}_train_normality_results.xlsx"
            )

            res.to_excel(out_path, index=False)

            fold_summary = summarize_one_fold(res)
            fold_summary.update({
                "Method": method,
                "Fold": fold,
                "Input_file": str(train_path),
                "Output_file": str(out_path)
            })

            fold_summary_records.append(fold_summary)
            all_results.append(res)

            print(
                f"{method} Fold {fold}: "
                f"tested={fold_summary['Tested_features']}, "
                f"non-normal={fold_summary['Non_normal_features']}"
            )

    # Aggregate all folds.
    all_results_df, feature_agg_df = aggregate_across_folds(all_results)

    fold_summary_df = pd.DataFrame(fold_summary_records)

    all_results_path = OUT_DIR / "all_fold_normality_results.xlsx"

    feature_agg_path = (
        OUT_DIR
        / "feature_normality_summary_across_folds.xlsx"
    )

    fold_summary_path = OUT_DIR / "fold_normality_summary.xlsx"

    all_results_df.to_excel(all_results_path, index=False)
    feature_agg_df.to_excel(feature_agg_path, index=False)
    fold_summary_df.to_excel(fold_summary_path, index=False)

    print("\nAll normality tests completed.")
    print(f"Detailed fold-wise results: {all_results_path}")
    print(f"Feature-level summary across folds: {feature_agg_path}")
    print(f"Fold-level summary: {fold_summary_path}")

    # Print a concise fold-level summary.
    print("\nFold-level summary:")
    print(
        fold_summary_df[
            [
                "Method",
                "Fold",
                "Total_features",
                "Tested_features",
                "Normal_like_features",
                "Non_normal_features",
                "Non_normal_percent_among_tested"
            ]
        ]
    )


# ============================================================
# 3. Program entry point
# ============================================================

if __name__ == "__main__":
    main()